In [1]:
# Install pip packages
!mkdir ~/tmpdir
!TMPDIR=~/tmpdir python3 -m pip install --upgrade --no-cache-dir "sagemaker<3" pandas numpy matplotlib seaborn scikit-learn "torch==2.6" torchvision boto3

mkdir: cannot create directory ‘/home/ec2-user/tmpdir’: File exists
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.


In [2]:
# Prepare dataset
%cd ~/SageMaker/
! rm -rf dataset/ dataset.zip
!wget -nc -q -O "dataset.zip" "https://www.kaggle.com/api/v1/datasets/download/zalando-research/fashionmnist"

!mkdir dataset -p
!unzip -q -o ./dataset.zip -d ./dataset/
# !mv ./dataset/cifar10/* ./dataset/
# !rm -r ./dataset/cifar10 dataset.zip

# %cd ./dataset/
# !zip -rq ../dataset.zip .
# %cd ..

/home/ec2-user/SageMaker


In [15]:
# Import pip packages
import os
import struct

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from IPython.display import display
from sagemaker.inputs import TrainingInput
from sagemaker.s3 import S3Uploader

In [4]:
# Set up SageMaker environment
import sagemaker

sess = sagemaker.Session()
bucket = sess.default_bucket()
prefix = "sagemaker"

from sagemaker import get_execution_role

role = get_execution_role()

In [5]:
# Upload dataset zip file to S3
inputs = S3Uploader.upload("dataset.zip", "s3://{}/{}".format(bucket, prefix))

In [32]:
class CustomDataset(Dataset):
    def __init__(self, images, labels=None, transform=None):
        self.images    = images
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]

        if self.transform:
            img = self.transform(img)

        if self.labels is not None:
            return img, int(self.labels[idx])
        return img

In [12]:
with open(f'./dataset/train-images-idx3-ubyte', 'rb') as f:
    _, n, rows, cols = struct.unpack('>IIII', f.read(16))
    train_images = np.fromfile(f, dtype=np.uint8).reshape(n, rows, cols, 1)

with open(f'./dataset/train-labels-idx1-ubyte', 'rb') as f:
    struct.unpack('>II', f.read(8))
    train_labels = np.fromfile(f, dtype=np.uint8)

with open(f'./dataset/t10k-images-idx3-ubyte', 'rb') as f:
    _, n, rows, cols = struct.unpack('>IIII', f.read(16))
    test_images = np.fromfile(f, dtype=np.uint8).reshape(n, rows, cols, 1)

with open(f'./dataset/t10k-labels-idx1-ubyte', 'rb') as f:
    struct.unpack('>II', f.read(8))
    test_labels = np.fromfile(f, dtype=np.uint8)

In [33]:
train_images = train_images.astype(np.float32) / 255.0
test_images  = test_images.astype(np.float32) / 255.0

train_dataset = CustomDataset(train_images, train_labels)
test_dataset = CustomDataset(test_images, test_labels)

# batch_size를 지정하는 Mini-batch 방식을 통해 학습, shuffle로 매 epoch마다 학습 데이터를 무작위 배치
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [21]:
print(len(train_dataset))  
print(len(test_dataset)) 

60000
10000


In [42]:
df_images = train_images.reshape(train_images.shape[0], -1)

df = pd.DataFrame(df_images)
df['label'] = train_labels

df.head()

,0,1,2,3,4,5,6,7,8,9,...,775,776,777,778,779,780,781,782,783,label
0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,0.0,0.0,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,9
1,0.0,0.0,0.0,0.0,0.0,3.637132e-15,0.0,0.0,0.000000e+00,0.000000e+00,...,4.146329e-13,4.728271e-13,2.764220e-13,0.0,0.0,0.0,0.0,0.0,0.0,0
2,0.0,0.0,0.0,0.0,0.0,0.000000e+00,0.0,0.0,0.000000e+00,8.001688e-14,...,0.000000e+00,3.637132e-15,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0
3,0.0,0.0,0.0,0.0,0.0,0.000000e+00,0.0,0.0,1.200253e-13,3.491646e-13,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,3
4,0.0,0.0,0.0,0.0,0.0,0.000000e+00,0.0,0.0,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0


In [25]:
from sagemaker.pytorch import PyTorch

estimator = PyTorch(
    entry_point="train.py",
    framework_version="2.6",
    py_version="py312",
    instance_type="ml.m6i.xlarge",
    instance_count=1,
    output_path="s3://{}/{}/output/".format(bucket, prefix),
    checkpoint_s3_uri="s3://{}/{}/checkpoint/".format(bucket, prefix),
    role=role,
    hyperparameters = {
        'epochs': 20,
        'batch_size': 64,
        'lr': 0.001,
        # 'backend': 'gloo'  # distributed
    },
    metric_definitions    = [
        {"Name": "train:loss", "Regex": r"Train Loss: ([0-9\.]+)"},
        {"Name": "train:acc",  "Regex": r"Train Acc: ([0-9\.]+)"},
        {"Name": "val:loss",   "Regex": r"Test Loss: ([0-9\.]+)"},
        {"Name": "val:acc",    "Regex": r"Test Acc: ([0-9\.]+)"},
    ],
)

estimator.fit({
    "train": TrainingInput("s3://{}/{}/dataset.zip".format(bucket, prefix)),  # /opt/ml/input/data/train
})

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: pytorch-training-2026-07-02-10-21-31-598


2026-07-02 10:21:33 Starting - Starting the training job...
2026-07-02 10:22:07 Downloading - Downloading input data...
2026-07-02 10:22:32 Downloading - Downloading the training image......
2026-07-02 10:23:12 Training - Training image download completed. Training in progress.bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
2026-07-02 10:23:22,357 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2026-07-02 10:23:22,357 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-07-02 10:23:22,358 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-07-02 10:23:22,366 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
2026-07-02 10:23:22,368 sagemaker_pytorch_container.training INFO     Invoking user training script.
2026-07-02 10:23:23,588 sagemaker-training-toolkit INFO     P

In [26]:
# Create new model + Endpoint Configuration
import boto3
import time

sm = boto3.client("sagemaker")

model = estimator.create_model(
  name = f'fashion-model-{int(time.time())}',
)
model_name = model.name
model._create_sagemaker_model(instance_type="ml.g5.xlarge")

new_config = f'fashion-endpoint-configuration-{int(time.time())}'
sm.create_endpoint_config(
    EndpointConfigName=new_config,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": model_name,
        "InitialInstanceCount": 1,
        "InstanceType": "ml.g5.xlarge"
    }]
)

INFO:sagemaker:Repacking model artifact (s3://sagemaker-us-east-1-200148130345/sagemaker/output/pytorch-training-2026-07-02-10-21-31-598/output/model.tar.gz), script artifact (s3://sagemaker-us-east-1-200148130345/pytorch-training-2026-07-02-10-21-31-598/source/sourcedir.tar.gz), and dependencies ([]) into single tar.gz file located at s3://sagemaker-us-east-1-200148130345/fashion-model-1782989123/model.tar.gz. This may take some time depending on model size...
INFO:sagemaker:Creating model with name: fashion-model-1782989123


{'EndpointConfigArn': 'arn:aws:sagemaker:us-east-1:200148130345:endpoint-config/fashion-endpoint-configuration-1782989124',
 'ResponseMetadata': {'RequestId': '997db0dd-a425-4ab3-8021-b8e4336451f7',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '997db0dd-a425-4ab3-8021-b8e4336451f7',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '122',
   'date': 'Thu, 02 Jul 2026 10:45:25 GMT'},
  'RetryAttempts': 0}}

In [27]:
# Create/Update Endpoint
# sm.create_endpoint(
#     EndpointName = "fashion-endpoint",
#     EndpointConfigName = new_config
# )

sm.update_endpoint(
    EndpointName = "fashion-endpoint",
    EndpointConfigName = new_config
)

{'EndpointArn': 'arn:aws:sagemaker:us-east-1:200148130345:endpoint/fashion-endpoint',
 'ResponseMetadata': {'RequestId': 'dab2987c-fd32-49b1-88f2-0c34e1d5816e',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'dab2987c-fd32-49b1-88f2-0c34e1d5816e',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '84',
   'date': 'Thu, 02 Jul 2026 10:47:13 GMT'},
  'RetryAttempts': 0}}

In [43]:
CLASSES = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
CLASSES

['T-shirt',
 'Trouser',
 'Pullover',
 'Dress',
 'Coat',
 'Sandal',
 'Shirt',
 'Sneaker',
 'Bag',
 'Ankle boot']

In [47]:
# Inference test
import base64, json, boto3

client = boto3.client("sagemaker-runtime")
endpoint_name = 'fashion-endpoint'
image_path = "pullover.png"

with open(image_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("utf-8")
response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",   #           application/json | text/plain | image/png
    Body=json.dumps({"image": b64}),  # json.dumps({"image": b64}) |        b64 |  f.read()
)

body = json.load(response["Body"])
print(body)
print(CLASSES[body["class_idx"]])

{'class_idx': 2, 'confidence': 0.8010050058364868}
Pullover


In [48]:
# Inference test
import base64, json, boto3

client = boto3.client("sagemaker-runtime")
endpoint_name = 'fashion-endpoint'
image_path = "shirt.jpg"

with open(image_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("utf-8")
response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",   #           application/json | text/plain | image/png
    Body=json.dumps({"image": b64}),  # json.dumps({"image": b64}) |        b64 |  f.read()
)

body = json.load(response["Body"])
print(body)
print(CLASSES[body["class_idx"]])

{'class_idx': 6, 'confidence': 0.5620474219322205}
Shirt
